In [1]:
!pip install sympy
!pip install opt_einsum
!pip install taichi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 20.2 MB/s eta 0:00:00


In [2]:
"""
MechDSL MVP: LaTeX -> SymPy -> SymPDE -> opt_einsum -> Taichi
A minimal working example of the computational mechanics compiler backend.
"""

import sympy as sp
import opt_einsum as oe
import numpy as np
import taichi as ti

# Initialize Taichi for GPU execution
ti.init(arch=ti.gpu)

# =============================================================================
# STAGE 1 & 2: Frontend Mock & SymPy -> SymPDE Mapper
# =============================================================================

def mock_nrpylatex_output():
    """Mocks the output of NRPyLaTeX parsing the elasticity DSL."""
    lmbda, mu = sp.symbols('lambda mu')
    return lmbda, mu

def build_sympde_mapper(domain, trial_func, test_func):
    """
    Creates a dictionary to map generic SymPy symbols to SymPDE objects.
    (Conceptual logic based on our design)
    """
    x, y = sp.symbols('x y')
    u_0 = sp.Function('u_0')(x, y)
    u_1 = sp.Function('u_1')(x, y)

    # In a full implementation, sympde.calculus.dx, dy are used here
    mapping = {
        x: domain.coordinates[0],
        y: domain.coordinates[1],
        u_0: trial_func[0],
        u_1: trial_func[1],
        sp.Derivative(u_0, x): "dx(trial_func[0])", # Mocked for MVP
        sp.Derivative(u_0, y): "dy(trial_func[0])",
    }
    return mapping

# =============================================================================
# STAGE 3 & 4: The Code Emitter (opt_einsum -> Taichi)
# =============================================================================

def emit_optimized_taichi_einsum(einsum_str, operand_names, shapes_dict, unroll_threshold=4, max_unrolled_lines=64):
    """
    Translates an opt_einsum path into flattened Taichi code.
    Uses an unroll_threshold to balance compile-time unrolling (registers)
    with dynamic loops (local memory) to prevent instruction cache explosion.
    An additional max_unrolled_lines guardrail prevents combinatorial explosion
    when multiple small dimensions are nested.
    """
    in_str, out_str = einsum_str.split('->')
    in_subs = [s.strip() for s in in_str.split(',')]
    out_sub = out_str.strip()

    # Get optimal path using dummy numpy arrays
    dummy_arrays = [np.ones([shapes_dict[c] for c in sub]) for sub in in_subs]
    path, path_info = oe.contract_path(einsum_str, *dummy_arrays, optimize='optimal')

    current_ops = list(operand_names)
    current_subs = list(in_subs)
    code = [
        "@ti.func",
        f"def compute_local_stiffness({', '.join(operand_names)}):"
    ]

    for step_num, step in enumerate(path):
        idx_b, idx_a = max(step), min(step)
        op_b, sub_b = current_ops.pop(idx_b), current_subs.pop(idx_b)
        op_a, sub_a = current_ops.pop(idx_a), current_subs.pop(idx_a)

        remaining_subs = "".join(current_subs) + out_sub
        new_sub = "".join(sorted(set(sub_a + sub_b) & set(remaining_subs)))

        out_name = f"temp_{step_num}" if step_num < len(path) - 1 else "K_local"

        flat_size = " * ".join([str(shapes_dict[c]) for c in new_sub]) if new_sub else "1"
        code.append(f"    # Step {step_num}: {op_a} ({sub_a}) @ {op_b} ({sub_b}) -> {out_name} ({new_sub})")
        if new_sub:
            code.append(f"    {out_name} = ti.Vector.zero(ti.f32, {flat_size})")
        else:
            code.append(f"    {out_name} = 0.0")

        all_chars = sorted(set(sub_a + sub_b))
        indent = "    "
        current_unroll_factor = 1

        for c in all_chars:
            dim_size = shapes_dict[c]
            # --- THE BALANCING LOGIC WITH GUARDRAIL ---
            if dim_size <= unroll_threshold and (current_unroll_factor * dim_size <= max_unrolled_lines):
                code.append(f"{indent}for {c} in ti.static(range({dim_size})):")
                current_unroll_factor *= dim_size
            else:
                code.append(f"{indent}for {c} in range({dim_size}):")
            indent += "    "

        def flat_idx(sub):
            if not sub: return "0"
            strides = []
            for i, c in enumerate(sub):
                stride = " * ".join([str(shapes_dict[x]) for x in sub[i+1:]])
                strides.append(f"({c} * {stride})" if stride else c)
            return " + ".join(strides)

        idx_out = flat_idx(new_sub)
        idx_a_str = flat_idx(sub_a)
        idx_b_str = flat_idx(sub_b)

        acc_str = f"{out_name}[{idx_out}]" if new_sub else out_name
        op_a_str = f"{op_a}[{idx_a_str}]" if sub_a else op_a
        op_b_str = f"{op_b}[{idx_b_str}]" if sub_b else op_b

        code.append(f"{indent}{acc_str} += {op_a_str} * {op_b_str}\n")

        current_ops.append(out_name)
        current_subs.append(new_sub)

    code.append(f"    return {out_name}")
    return "\n".join(code)

# =============================================================================
# STAGE 5: Global Assembly & Execution Setup
# =============================================================================

def demonstrate_compiler():
    print("--- MechDSL Compilation Pipeline ---")

    # 1. Define the elasticity tensors for a 2D Quad element
    # q: quadrature points (4)
    # i, j: spatial dimensions (3 in Voigt notation)
    # a, b: local degrees of freedom (8 for a quad4 element)
    shapes = {'q': 4, 'i': 3, 'a': 8, 'j': 3, 'b': 8}

    print("\n[1/3] Running Optimization Pass (opt_einsum)...")
    # Setting threshold to 3.
    # This forces spatial dims (i,j=3) to unroll into registers,
    # but uses dynamic loops for nodes (a,b=8) and quad points (q=4).
    # max_unrolled_lines prevents combinatorial explosion across loops.
    generated_code = emit_optimized_taichi_einsum(
        einsum_str="q, qia, ij, qjb -> ab",
        operand_names=["W", "B_left", "D", "B_right"],
        shapes_dict=shapes,
        unroll_threshold=3,
        max_unrolled_lines=64
    )

    print("\n[2/3] Emitted Taichi Code:")
    print("-" * 50)
    print(generated_code)
    print("-" * 50)

    print("\n[3/3] Compiling JIT Kernel and Setting up Assembly...")
    # Dynamically load the generated function into the Python environment
    exec(generated_code, globals())

    # --- Dummy Mesh Data for Assembly Demo ---
    num_nodes = 4
    num_elements = 1
    dim = 2

    vertices = ti.Vector.field(dim, dtype=ti.f32, shape=num_nodes)
    elements = ti.Vector.field(4, dtype=ti.i32, shape=num_elements)
    is_dirichlet = ti.field(dtype=ti.i32, shape=(num_nodes, dim))

    # Mock some basic data
    elements[0] = [0, 1, 2, 3]
    is_dirichlet[0, 0] = 1 # Constrain node 0, x-direction

    # Define the Global Assembly Kernel using the dynamically generated function
    @ti.kernel
    def assemble_global_system(
        W_in: ti.types.ndarray(),
        B_in: ti.types.ndarray(),
        D_in: ti.types.ndarray()
    ):
        # In a real scenario, W, B, D are populated from shape functions.
        # Here we map the ndarrays to flat ti.Vectors for the emitted function.
        W = ti.Vector.zero(ti.f32, 4)
        B = ti.Vector.zero(ti.f32, 4 * 3 * 8)
        D = ti.Vector.zero(ti.f32, 3 * 3)

        for e in elements:
            # Call our dynamically generated Taichi function
            Ke = compute_local_stiffness(W, B, D, B)

            # --- Dirichlet Enforcement & Assembly Logic ---
            n0, n1, n2, n3 = elements[e]
            node_indices = [n0, n1, n2, n3]

            for i in ti.static(range(4)):
                for j in ti.static(range(4)):
                    for d1 in ti.static(range(dim)):
                        for d2 in ti.static(range(dim)):
                            g_row = node_indices[i]
                            g_col = node_indices[j]
                            row = g_row * dim + d1
                            col = g_col * dim + d2

                            # Flat index lookup for Ke (8x8 -> 64)
                            val = Ke[(i * dim + d1) * 8 + (j * dim + d2)]

                            # Node Masking for Dirichlet BCs
                            if is_dirichlet[g_row, d1] == 1:
                                val = 1.0 if row == col else 0.0
                            elif is_dirichlet[g_col, d2] == 1:
                                val = 0.0

                            # (Here we would scatter to K_global sparse builder)
                            # K_global.print_triplet(row, col, val)

    print("Assembly kernel successfully compiled!")
    print("Pipeline Ready for execution.")

if __name__ == "__main__":
    demonstrate_compiler()

[Taichi] version 1.7.4, llvm 15.0.4, commit b4b956fd, linux, python 3.12.13
[Taichi] Starting on arch=cuda
--- MechDSL Compilation Pipeline ---

[1/3] Running Optimization Pass (opt_einsum)...

[2/3] Emitted Taichi Code:
--------------------------------------------------
@ti.func
def compute_local_stiffness(W, B_left, D, B_right):
    # Step 0: W (q) @ D (ij) -> temp_0 (ijq)
    temp_0 = ti.Vector.zero(ti.f32, 3 * 3 * 4)
    for i in ti.static(range(3)):
        for j in ti.static(range(3)):
            for q in range(4):
                temp_0[(i * 3 * 4) + (j * 4) + q] += W[q] * D[(i * 3) + j]

    # Step 1: B_left (qia) @ temp_0 (ijq) -> temp_1 (ajq)
    temp_1 = ti.Vector.zero(ti.f32, 8 * 3 * 4)
    for a in range(8):
        for i in ti.static(range(3)):
            for j in ti.static(range(3)):
                for q in range(4):
                    temp_1[(a * 3 * 4) + (j * 4) + q] += B_left[(q * 3 * 8) + (i * 8) + a] * temp_0[(i * 3 * 4) + (j * 4) + q]

    # Step 2: B_right (qj

In [3]:
demonstrate_compiler()

--- MechDSL Compilation Pipeline ---

[1/3] Running Optimization Pass (opt_einsum)...

[2/3] Emitted Taichi Code:
--------------------------------------------------
@ti.func
def compute_local_stiffness(W, B_left, D, B_right):
    # Step 0: W (q) @ D (ij) -> temp_0 (ijq)
    temp_0 = ti.Vector.zero(ti.f32, 3 * 3 * 4)
    for i in ti.static(range(3)):
        for j in ti.static(range(3)):
            for q in range(4):
                temp_0[(i * 3 * 4) + (j * 4) + q] += W[q] * D[(i * 3) + j]

    # Step 1: B_left (qia) @ temp_0 (ijq) -> temp_1 (ajq)
    temp_1 = ti.Vector.zero(ti.f32, 8 * 3 * 4)
    for a in range(8):
        for i in ti.static(range(3)):
            for j in ti.static(range(3)):
                for q in range(4):
                    temp_1[(a * 3 * 4) + (j * 4) + q] += B_left[(q * 3 * 8) + (i * 8) + a] * temp_0[(i * 3 * 4) + (j * 4) + q]

    # Step 2: B_right (qjb) @ temp_1 (ajq) -> K_local (ab)
    K_local = ti.Vector.zero(ti.f32, 8 * 8)
    for a in range(8):
      